In [1]:
import subprocess, re

SBATCH = "/projects/a5u/adu_dev/aisi-economy-index/nbs/AISI_demo/Stage3_negative_selection/dev/sbatch/run_llama_drops_fix_test.sbatch"

out = subprocess.check_output(["sbatch", SBATCH], text=True)
jobid = re.search(r"(\d+)", out).group(1)

print("JOBID:", jobid)


JOBID: 2095000


In [2]:
import time, subprocess
from pathlib import Path

LOG_DIR = Path("/projects/a5u/adu_dev/aisi-economy-index/logs")

def sh(cmd):
    return subprocess.run(cmd, shell=True, text=True, capture_output=True).stdout.strip()

while True:
    print("\n=== squeue ===")
    sq = sh(f"squeue -j {jobid}")
    print(sq if sq else "<finished>")

    out_logs = sorted(LOG_DIR.glob(f"*{jobid}*.out"))
    err_logs = sorted(LOG_DIR.glob(f"*{jobid}*.err"))

    if out_logs:
        print("\n--- STDOUT ---")
        print(sh(f"tail -n 20 {out_logs[-1]}"))

    if err_logs:
        print("\n--- STDERR ---")
        print(sh(f"tail -n 20 {err_logs[-1]}"))

    if not sq:
        break

    time.sleep(5)



=== squeue ===
JOBID         USER PARTITION                     NAME ST TIME_LIMIT       TIME  TIME_LEFT NODES NODELIST(REASON)
2095000 autonomyluiz     workq    z_llama_drop_new_test  R    6:00:00       0:09    5:59:51     1 nid010881

--- STDOUT ---
[TASK] 0 -> adzuna_month01
[NPZ]  /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/llm_negative_selection/adzuna_month01.npz

--- STDERR ---


=== squeue ===
JOBID         USER PARTITION                     NAME ST TIME_LIMIT       TIME  TIME_LEFT NODES NODELIST(REASON)
2095000 autonomyluiz     workq    z_llama_drop_new_test  R    6:00:00       0:14    5:59:46     1 nid010881

--- STDOUT ---
[TASK] 0 -> adzuna_month01
[NPZ]  /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/llm_negative_selection/adzuna_month01.npz

--- STDERR ---


=== squeue ===
JOBID         USER PARTITION                     NAME ST TIME_LIMIT       TIME  TI

KeyboardInterrupt: 

In [3]:
import json
import re
import random
import statistics
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime

import numpy as np


# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------
_RUN_RE = re.compile(
    r"llama_drop_only_(?P<month>adzuna_month\d{2})_(?P<start>\d+)_(?P<stop>\d+)_job(?P<jobid>\d+)_task(?P<taskid>\d+)_\d{8}_\d{6}\.jsonl$"
)

IT_PATTERNS = [
    r"\bsoftware\b", r"\bdeveloper\b", r"\bengineer\b", r"\bdata\b", r"\bml\b", r"\bai\b",
    r"\bcloud\b", r"\bcyber\b", r"\bsecurity\b", r"\bnetwork\b", r"\bsystems?\b",
    r"\bdatabase\b", r"\bit\b", r"\bdevops\b"
]
_IT_RE = re.compile("|".join(IT_PATTERNS), flags=re.I)

def _is_it_role(title: str) -> bool:
    if not title:
        return False
    return bool(_IT_RE.search(title))


def _infer_npz_from_jsonl(jsonl_path: Path) -> Path:
    """
    Your runs are like:
      .../llm_negative_selection/adzuna_month01/llama_drop_only_adzuna_month01_0_1000_job...jsonl
    And the matching NPZ (context) is:
      .../llm_negative_selection/adzuna_month01.npz   (or stage_3 equivalent if you moved it)
    """
    m = _RUN_RE.search(jsonl_path.name)
    if not m:
        raise ValueError(f"JSONL filename doesn't match expected pattern: {jsonl_path.name}")

    month = m.group("month")

    # Prefer same parent root: .../llm_negative_selection/<month>/  -> .../llm_negative_selection/<month>.npz
    # i.e. go up one level (month folder) then pick month.npz
    parent = jsonl_path.parent.parent  # .../llm_negative_selection
    npz_path = parent / f"{month}.npz"
    return npz_path


def _load_npz_lookup(npz_path: Path):
    """
    NPZ must contain:
      job_id, job_ad_title, job_desc, job_tasks, domain, job_sector_category
    Optional:
      job_description
    """
    with np.load(npz_path, allow_pickle=True) as z:
        # required
        job_id = z["job_id"].astype(str)
        job_ad_title = z["job_ad_title"]
        job_desc = z["job_desc"]
        job_tasks = z["job_tasks"]
        domain = z["domain"] if "domain" in z.files else None
        job_sector_category = z["job_sector_category"] if "job_sector_category" in z.files else None
        job_description = z["job_description"] if "job_description" in z.files else None

    lookup = {}
    n = len(job_id)
    for i in range(n):
        jid = job_id[i]
        lookup[jid] = {
            "job_ad_title": None if job_ad_title[i] is None else str(job_ad_title[i]),
            "job_desc": None if job_desc[i] is None else str(job_desc[i]),
            "job_tasks": None if job_tasks[i] is None else str(job_tasks[i]),
            "job_description": None if job_description is None or job_description[i] is None else str(job_description[i]),
            "domain": None if domain is None or domain[i] is None else str(domain[i]),
            "job_sector_category": None if job_sector_category is None or job_sector_category[i] is None else str(job_sector_category[i]),
        }
    return lookup


# ---------------------------------------------------------------------
# Main: report generator
# ---------------------------------------------------------------------
def gen_report(jsonl_path: str, *, npz_path: str | None = None, sample_n: int = 30, seed: int = 0):
    """
    jsonl_path: path to llama_drop_only_*.jsonl
    npz_path  : optional explicit context NPZ (if you moved files). If None, inferred from jsonl name/dir.
    """
    random.seed(seed)

    jsonl_path = Path(jsonl_path)
    if not jsonl_path.exists():
        raise FileNotFoundError(f"Missing JSONL: {jsonl_path}")

    inferred_npz = _infer_npz_from_jsonl(jsonl_path)
    npz_path = Path(npz_path) if npz_path else inferred_npz
    if not npz_path.exists():
        raise FileNotFoundError(f"Missing NPZ: {npz_path}")

    # report folder: alongside month folder, inside evaluation_reports/
    out_root = jsonl_path.parent.parent  # .../llm_negative_selection
    report_dir = out_root / "evaluation_reports"
    report_dir.mkdir(parents=True, exist_ok=True)

    print("JSONL:", jsonl_path)
    print("NPZ :", npz_path)
    print("OUT :", report_dir)

    lookup = _load_npz_lookup(npz_path)

    before_counts = []
    after_counts = []
    kept_titles = Counter()
    domain_kept = defaultdict(list)

    it_leak = 0
    total_kept = 0
    empty_outputs = 0

    samples = []
    seen_sample_ids = set()

    with jsonl_path.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                r = json.loads(line)
            except Exception:
                continue

            jid = str(r.get("job_id", ""))
            if not jid:
                continue

            ctx = lookup.get(jid, {})

            cand = r.get("candidates") or []
            final = r.get("final") or []

            before_counts.append(len(cand))
            after_counts.append(len(final))

            if len(final) == 0:
                empty_outputs += 1

            for t in final:
                kept_titles[t] += 1
                total_kept += 1
                if _is_it_role(t):
                    it_leak += 1

            dom = ctx.get("domain") or "UNKNOWN"
            domain_kept[dom].append(len(final))

            # light sampling: keep the first sample_n unique job_ids with a random gate
            if len(samples) < sample_n and jid not in seen_sample_ids and random.random() < 0.05:
                seen_sample_ids.add(jid)
                samples.append({
                    "job_id": jid,
                    "job_ad_title": ctx.get("job_ad_title"),
                    "domain": ctx.get("domain"),
                    "sector": ctx.get("job_sector_category"),
                    "job_desc": ctx.get("job_desc"),
                    "kept": final,
                    "dropped": r.get("drop") or [],
                })

    if not before_counts:
        raise RuntimeError("No valid rows parsed from JSONL.")

    before_avg = statistics.mean(before_counts)
    after_avg = statistics.mean(after_counts)
    drop_rate = 1.0 - (after_avg / before_avg if before_avg else 0.0)
    it_share = it_leak / max(total_kept, 1)

    metrics = {
        "jobs": len(before_counts),
        "avg_candidates_before": round(before_avg, 3),
        "avg_candidates_after": round(after_avg, 3),
        "drop_rate": round(drop_rate, 4),
        "empty_outputs_percent": round(100.0 * empty_outputs / len(before_counts), 3),
        "it_leakage_share": round(it_share, 4),
        "min_kept": int(min(after_counts)),
        "max_kept": int(max(after_counts)),
    }

    domain_summary = {d: round(statistics.mean(v), 3) for d, v in domain_kept.items()}

    report = {
        "run": {
            "jsonl": str(jsonl_path),
            "npz": str(npz_path),
            "generated_at": datetime.now().isoformat(timespec="seconds"),
        },
        "global_metrics": metrics,
        "top_kept_roles": kept_titles.most_common(25),
        "domain_summary_avg_kept": dict(sorted(domain_summary.items(), key=lambda x: (-x[1], x[0]))),
        "sample_cases": samples,
    }

    # stable report names based on JSONL stem
    base = jsonl_path.stem
    report_json = report_dir / f"{base}_report.json"
    report_txt  = report_dir / f"{base}_report.txt"

    with report_json.open("w", encoding="utf-8") as f:
        json.dump(report, f, indent=2)

    with report_txt.open("w", encoding="utf-8") as f:
        f.write("=== GLOBAL METRICS ===\n")
        for k, v in metrics.items():
            f.write(f"{k}: {v}\n")

        f.write("\n=== TOP KEPT OCCUPATIONS ===\n")
        for title, cnt in kept_titles.most_common(25):
            f.write(f"{cnt}  {title}\n")

        f.write("\n=== DOMAIN AVG KEPT ===\n")
        for d, v in sorted(domain_summary.items(), key=lambda x: (-x[1], x[0])):
            f.write(f"{d}: {v}\n")

        f.write("\n=== SAMPLE CASES (truncated) ===\n")
        for s in samples[:10]:
            f.write(f"\njob_id: {s['job_id']}\n")
            f.write(f"title:  {s.get('job_ad_title')}\n")
            f.write(f"domain: {s.get('domain')} | sector: {s.get('sector')}\n")
            f.write(f"kept:   {s.get('kept')}\n")
            f.write(f"drop:   {s.get('dropped')}\n")

    print("\nSaved:")
    print(" ", report_json)
    print(" ", report_txt)

    return report_json, report_txt


In [ ]:
/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/llm_negative_selection/adzuna_month01/llama_drop_only_adzuna_month01_0_1000_job2095000_task0_20260201_015704.jsonl

In [5]:
#[DONE] wrote: /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/llm_negative_selection/adzuna_month01/llama_drop_only_adzuna_month01_0_1000_job2095000_task0_20260201_015704.jsonl
gen_report(
    "/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/"
    "adzuna_2022_all_months/llm_negative_selection/adzuna_month01/"
    "llama_drop_only_adzuna_month01_0_1000_job2095000_task0_20260201_015704.jsonl"
)


JSONL: /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/llm_negative_selection/adzuna_month01/llama_drop_only_adzuna_month01_0_1000_job2095000_task0_20260201_015704.jsonl
NPZ : /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/llm_negative_selection/adzuna_month01.npz
OUT : /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/llm_negative_selection/evaluation_reports

Saved:
  /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/llm_negative_selection/evaluation_reports/llama_drop_only_adzuna_month01_0_1000_job2095000_task0_20260201_015704_report.json
  /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/llm_negative_selection/evaluation_reports/llama_drop_only_adzuna_month01_0_1000_job2095000_task0_20260201_015704_report.

(PosixPath('/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/llm_negative_selection/evaluation_reports/llama_drop_only_adzuna_month01_0_1000_job2095000_task0_20260201_015704_report.json'),
 PosixPath('/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/llm_negative_selection/evaluation_reports/llama_drop_only_adzuna_month01_0_1000_job2095000_task0_20260201_015704_report.txt'))

In [8]:

from pathlib import Path
p = Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/llm_negative_selection/evaluation_reports/")
latest = max(p.glob("llama_drop_only_adzuna_month01_0_1000_job2095000_task0_20260201_015704_report.json"), key=lambda x: x.stat().st_mtime)
print(latest.read_text())


{
  "run": {
    "jsonl": "/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/llm_negative_selection/adzuna_month01/llama_drop_only_adzuna_month01_0_1000_job2095000_task0_20260201_015704.jsonl",
    "npz": "/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/llm_negative_selection/adzuna_month01.npz",
    "generated_at": "2026-02-01T02:02:00"
  },
  "global_metrics": {
    "jobs": 1000,
    "avg_candidates_before": 8.517,
    "avg_candidates_after": 3.706,
    "drop_rate": 0.5649,
    "empty_outputs_percent": 0.0,
    "it_leakage_share": 0.0742,
    "min_kept": 1,
    "max_kept": 5
  },
  "top_kept_roles": [
    [
      "Personal Care Aides",
      56
    ],
    [
      "Sales Managers",
      54
    ],
    [
      "Customer Service Representatives",
      51
    ],
    [
      "Information Technology Project Managers",
      47
    ],
    [
      "Social and Human Service Assista

Ready to scale: my call

Yes, with one caveat:

The “domains at 5” pattern suggests you’ll still get slot-filling in some areas.

That’s not a blocker for scaling, but it means you should expect some noise and do a light post-trim or prompt tweak later.

Practical scaling checklist (fast)

Before you burn big GPU:

Run one full month shard (not 1k) and re-check:

empty_outputs stays ~0

it_leakage_share stays <10%

avg_after stays ~3–4 (not drifting to 5)

Sample 50 cases from:

domains that used to be noisy

the top 10 kept occupations

If it holds, go wide.

Bottom line: this is a clear step up vs yesterday and it’s good enough to scale now.